# **WEEK 7 — Nonlinear Dynamics, CCM, and Modern Methods**

>Jerald B. Bongalos | PhD in Data Science | Asian Institute of Management

---

*References:*
>Sugihara, G. & May, R. M. (1990). Nonlinear forecasting as a way of distinguishing chaos from measurement error in time series. *Nature*, 344, 734–741.
>
>Sugihara, G. et al. (2012). Detecting causality in complex ecosystems. *Science*, 338, 496–500.
>
>Shumway, R. H., & Stoffer, D. S. *Time Series Analysis and Its Applications* (Supplementary topics)

---

**Purpose of this Notebook**

This notebook moves beyond linear models into **nonlinear dynamics** and **dynamical causality**.
Where ARMA assumes linear structure and Granger assumes stochastic separability,
the methods here exploit the **geometry of attractors** reconstructed from time series.

**The goal is to understand:**

- how deterministic chaos generates time series that look random,
- how Takens' theorem justifies reconstructing dynamics from a single observed variable,
- how simplex projection and S-maps perform nonlinear forecasting,
- how Convergent Cross-Mapping (CCM) detects causality in coupled dynamical systems,
- and how these methods compare to linear approaches (Granger, ARMA).

**Learning Outcomes**

By the end of this notebook, you should be able to:

1. Explain Takens' embedding theorem and its significance
2. Reconstruct a shadow attractor using time-delay embedding
3. Apply simplex projection for nonlinear forecasting and determine optimal embedding dimension
4. Use S-maps to detect nonlinearity (linearity test via $\theta$ parameter)
5. Implement and interpret CCM for dynamical causality
6. Distinguish Granger causality from CCM and explain when each is appropriate
7. Describe the conceptual role of modern ML methods (RNN/LSTM) in time series

---

**Notebook Structure**

| Part | Topic | Type | Priority |
|------|-------|------|----------|
| 1 | Deterministic Chaos and Time Series | Theory + Simulation | HIGH |
| 2 | Takens' Theorem and Embedding | Theory + Simulation | HIGH |
| 3 | Simplex Projection (Nonlinear Forecasting) | Theory + Simulation | HIGH |
| 4 | S-maps (Nonlinearity Detection) | Theory + Simulation | HIGH |
| 5 | Convergent Cross-Mapping (CCM) | Theory + Simulation | CRITICAL |
| 6 | Granger vs CCM: When to Use Which | Theory | CRITICAL |
| 7 | Modern ML Methods: A Brief Overview | Theory | MEDIUM |
| 8 | Synthesis + Self-Test | Summary + Exam Prep | — |


In [ ]:
# ============================================================
# SETUP
# ============================================================
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.spatial.distance import cdist
from scipy.stats import pearsonr
from mpl_toolkits.mplot3d import Axes3D

np.random.seed(123)

# --- Lorenz system simulator ---
def simulate_lorenz(n=5000, dt=0.01, sigma=10, rho=28, beta=8/3,
                    x0=1.0, y0=1.0, z0=1.0, skip=500):
    """Simulate Lorenz attractor via Euler integration."""
    N = n + skip
    x, y, z = np.zeros(N), np.zeros(N), np.zeros(N)
    x[0], y[0], z[0] = x0, y0, z0
    for t in range(N - 1):
        x[t+1] = x[t] + dt * sigma * (y[t] - x[t])
        y[t+1] = y[t] + dt * (x[t] * (rho - z[t]) - y[t])
        z[t+1] = z[t] + dt * (x[t] * y[t] - beta * z[t])
    return x[skip:], y[skip:], z[skip:]

# --- Logistic map ---
def logistic_map(r=3.9, n=2000, x0=0.4, burnin=500):
    """Iterate x_{t+1} = r * x_t * (1 - x_t)."""
    N = n + burnin
    x = np.zeros(N)
    x[0] = x0
    for t in range(N - 1):
        x[t+1] = r * x[t] * (1 - x[t])
    return x[burnin:]

print("Setup complete.")


## **PART 1 — Deterministic Chaos and Time Series**

> **Why this part matters**
>
> All methods in Weeks 1–6 assume that unpredictability comes from **stochastic noise**.
> But some systems are **deterministic** yet appear random — this is **chaos**.
> Chaotic systems are sensitive to initial conditions: tiny differences grow exponentially,
> making long-term prediction impossible despite deterministic rules.
>
> Recognizing chaos matters because: linear models (ARMA) will fail on chaotic data,
> but **nonlinear methods** can exploit the underlying deterministic structure for short-term prediction.

---

### **1.1 Key Properties of Chaotic Systems**

- **Deterministic**: governed by fixed rules (no randomness)
- **Sensitive dependence on initial conditions**: nearby trajectories diverge exponentially
- **Bounded**: trajectories remain in a finite region (attractor)
- **Aperiodic**: never exactly repeats, but has structure

---

### **1.2 The Logistic Map**

$$
x_{t+1} = r\, x_t(1 - x_t), \qquad x_t \in [0, 1]
$$

For $r = 3.9$: deterministic, bounded, aperiodic — chaotic.

---

### **1.3 The Lorenz System**

$$
\dot{x} = \sigma(y - x), \qquad \dot{y} = x(\rho - z) - y, \qquad \dot{z} = xy - \beta z
$$

A 3D continuous-time system that produces the famous "butterfly" attractor.

> **Exam-safe statement:** "Chaotic systems are deterministic but practically unpredictable
> at long horizons due to sensitive dependence. The attractor has geometric structure
> that can be exploited for short-term nonlinear forecasting."


In [ ]:
# ============================================================
# PART 1: Chaos — Logistic Map + Lorenz Attractor
# ============================================================

# --- A) Logistic map ---
np.random.seed(123)
x_logistic = logistic_map(r=3.9, n=500)

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

axes[0].plot(x_logistic[:200], linewidth=0.8, color="#2c3e50")
axes[0].set_title("Logistic Map ($r=3.9$) — Time Series", fontsize=11)
axes[0].set_xlabel("$t$")
axes[0].grid(True, alpha=0.3)

# Lag plot (x_t vs x_{t+1})
axes[1].scatter(x_logistic[:-1], x_logistic[1:], s=2, alpha=0.5, color="#e74c3c")
axes[1].set_title("$x_{t+1}$ vs $x_t$ (deterministic parabola!)", fontsize=11)
axes[1].set_xlabel("$x_t$")
axes[1].set_ylabel("$x_{t+1}$")
axes[1].grid(True, alpha=0.3)

# ACF looks like noise
from statsmodels.tsa.stattools import acf
a = acf(x_logistic, nlags=30, fft=True)
ci = 1.96 / np.sqrt(len(x_logistic))
axes[2].stem(range(len(a)), a, basefmt=" ")
axes[2].axhline(ci, color="red", linestyle="--", alpha=0.5)
axes[2].axhline(-ci, color="red", linestyle="--", alpha=0.5)
axes[2].set_title("ACF (looks like white noise — but it's deterministic!)", fontsize=11)
axes[2].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# --- B) Lorenz attractor ---
lx, ly, lz = simulate_lorenz(n=5000, dt=0.01)

fig = plt.figure(figsize=(14, 5))
ax1 = fig.add_subplot(121, projection="3d")
ax1.plot(lx[::3], ly[::3], lz[::3], linewidth=0.3, color="#2c3e50")
ax1.set_title("Lorenz Attractor (3D)", fontsize=11)
ax1.set_xlabel("$x$"); ax1.set_ylabel("$y$"); ax1.set_zlabel("$z$")

ax2 = fig.add_subplot(122)
ax2.plot(lx[:1000], linewidth=0.5, color="#2c3e50")
ax2.set_title("Lorenz $x(t)$ — looks noisy, is deterministic", fontsize=11)
ax2.set_xlabel("$t$")
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("Key observations:")
print("  Logistic map: ACF looks like white noise, but lag plot reveals deterministic structure")
print("  Lorenz: time series looks random, but 3D attractor reveals geometric structure")
print("  ARMA models would treat these as noise — they would miss the structure entirely")


## **PART 2 — Takens' Theorem and Embedding**

> **Why this part matters**
>
> In practice, we observe **one variable** from a multi-dimensional dynamical system.
> Takens' theorem (1981) says we can **reconstruct the attractor geometry** from
> a single time series using time-delay embedding — and this reconstruction preserves
> the essential dynamical properties (topology, Lyapunov exponents).

---

### **2.1 Time-Delay Embedding**

Given a scalar time series $\{x_t\}$, construct vectors:

$$
\mathbf{v}_t = (x_t, x_{t-\tau}, x_{t-2\tau}, \ldots, x_{t-(E-1)\tau})
$$

where:
- $E$ = **embedding dimension** (number of coordinates)
- $\tau$ = **time delay** (lag between coordinates)

---

### **2.2 Takens' Theorem (Informal)**

> If $\{x_t\}$ is a generic observation of a $d$-dimensional dynamical system,
> then the time-delay embedding with $E \geq 2d + 1$ is a **diffeomorphism**
> (smooth, invertible map) from the original attractor to the reconstructed one.

**In practice:** the shadow attractor preserves:
- Topological structure (what is near stays near)
- Dynamical properties (Lyapunov exponents, dimension)
- **Prediction**: nearby points on the shadow attractor have nearby futures

---

### **2.3 Choosing $E$ and $\tau$**

- **$E$ (embedding dimension):** typically selected by optimizing out-of-sample forecast skill (simplex projection, Part 3)
- **$\tau$ (time delay):** first zero of ACF, or first minimum of mutual information; for many applications $\tau = 1$ works

> **Exam-critical:** "Takens' theorem justifies using time-delay embedding of a single
> observed variable to reconstruct the dynamical attractor. This is the foundation
> for simplex projection, S-maps, and CCM."


In [ ]:
# ============================================================
# PART 2: Time-Delay Embedding — Reconstructing Attractors
# ============================================================

def embed(x, E, tau=1):
    """Create time-delay embedding matrix. Rows = embedded vectors."""
    n = len(x)
    rows = n - (E - 1) * tau
    M = np.zeros((rows, E))
    for j in range(E):
        M[:, j] = x[j*tau : j*tau + rows]
    return M

# --- Lorenz: embed x(t) in 3D and compare to true attractor ---
lx_sub = lx[::5][:1000]  # subsample for clarity

fig, axes = plt.subplots(1, 3, figsize=(15, 5))

# True attractor projection (x vs z)
ax = axes[0]
ax.plot(lx[::5][:1000], lz[::5][:1000], linewidth=0.3, color="#2c3e50")
ax.set_title("True Lorenz: $x$ vs $z$", fontsize=11)
ax.set_xlabel("$x(t)$"); ax.set_ylabel("$z(t)$")
ax.grid(True, alpha=0.3)

# Shadow attractor E=2, tau=5
M2 = embed(lx_sub, E=2, tau=5)
ax = axes[1]
ax.plot(M2[:, 0], M2[:, 1], linewidth=0.3, color="#e74c3c")
ax.set_title("Shadow: $E=2$, $\\tau=5$", fontsize=11)
ax.set_xlabel("$x(t)$"); ax.set_ylabel("$x(t-5\\tau)$")
ax.grid(True, alpha=0.3)

# Shadow attractor E=3, tau=5
M3 = embed(lx_sub, E=3, tau=5)
ax = axes[2]
ax = fig.add_subplot(133, projection="3d")
ax.plot(M3[:, 0], M3[:, 1], M3[:, 2], linewidth=0.3, color="#27ae60")
ax.set_title("Shadow: $E=3$, $\\tau=5$", fontsize=11)

plt.tight_layout()
plt.show()

# --- Logistic map: E=2 recovers the parabola ---
x_log = logistic_map(r=3.9, n=2000)
M_log = embed(x_log, E=2, tau=1)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].scatter(M_log[:, 0], M_log[:, 1], s=1, alpha=0.4, color="#e74c3c")
axes[0].set_title("Logistic Map: $E=2$, $\\tau=1$", fontsize=11)
axes[0].set_xlabel("$x_t$"); axes[0].set_ylabel("$x_{t-1}$")
axes[0].grid(True, alpha=0.3)

M_log3 = embed(x_log, E=3, tau=1)
ax = fig.add_subplot(122, projection="3d")
ax.scatter(M_log3[:, 0], M_log3[:, 1], M_log3[:, 2], s=0.5, alpha=0.3, color="#8e44ad")
ax.set_title("Logistic Map: $E=3$, $\\tau=1$", fontsize=11)

plt.tight_layout()
plt.show()

print("Observations:")
print("  Lorenz: shadow attractor from x(t) alone recovers butterfly shape")
print("  Logistic: E=2 recovers the parabola; E=3 maps to a curve in 3D")
print("  This is Takens' theorem in action: single variable -> full attractor geometry")


## **PART 3 — Simplex Projection (Nonlinear Forecasting)**

> **Why this part matters**
>
> Simplex projection (Sugihara & May, 1990) forecasts by finding
> **nearest neighbors on the reconstructed attractor** and tracking where they go.
> It exploits the geometric structure that linear models miss entirely.

---

### **3.1 The Algorithm**

1. Embed the time series in $E$ dimensions: $\mathbf{v}_t = (x_t, x_{t-1}, \ldots, x_{t-E+1})$
2. To predict $x_{T+1}$ from $\mathbf{v}_T$, find the $E+1$ **nearest neighbors** of $\mathbf{v}_T$ in the library (past embedded vectors)
3. Track where each neighbor goes one step ahead
4. Predict $\hat{x}_{T+1}$ as a **weighted average** of the neighbors' futures (weights decay with distance)

---

### **3.2 Choosing $E$: Embedding Dimension**

The optimal $E$ is selected by maximizing **out-of-sample forecast correlation** ($\rho$):

- Sweep $E = 1, 2, \ldots, E_{\max}$
- For each $E$, run simplex projection on a train/test split
- Select $E$ that maximizes $\rho(\hat{x}, x)$ on the test set

The optimal $E$ approximates the **dimensionality of the attractor**.

---

### **3.3 Key Insight: Chaos vs Noise**

| Property | Chaos | Stochastic noise |
|---|---|---|
| Simplex $\rho$ at $E_{\text{opt}}$ | High (short-term predictable) | Low |
| $\rho$ decay with horizon $h$ | Rapid (sensitive dependence) | Gradual (linear decay) |
| Optimal $E$ | Low (finite attractor dimension) | $E$ has no clear optimum |

> **Exam-safe statement:** "Simplex projection uses nearest neighbors on the reconstructed
> attractor for nonlinear forecasting. The optimal embedding dimension $E$ reflects
> attractor dimensionality, and forecast skill decay rate distinguishes chaos from noise."


In [ ]:
# ============================================================
# PART 3: Simplex Projection — From Scratch
# ============================================================

def simplex_projection(x, E, tp=1, lib_range=None, pred_range=None):
    """
    Simplex projection (Sugihara & May 1990).
    x: time series (1D array)
    E: embedding dimension
    tp: prediction horizon (time steps ahead)
    lib_range, pred_range: (start, end) indices for library and prediction sets
    Returns: (observed, predicted) arrays for the prediction set.
    """
    M = embed(x, E + 1, tau=1)  # E+1 to have target column
    n_emb = M.shape[0]

    if lib_range is None:
        lib_range = (0, n_emb // 2)
    if pred_range is None:
        pred_range = (n_emb // 2, n_emb - tp)

    # Library vectors (E dims) and their tp-step-ahead targets
    lib_idx = np.arange(lib_range[0], min(lib_range[1], n_emb - tp))
    pred_idx = np.arange(pred_range[0], min(pred_range[1], n_emb - tp))

    lib_vecs = M[lib_idx, :E]
    lib_targets = x[lib_idx + E - 1 + tp]  # value tp steps after the last coord

    pred_vecs = M[pred_idx, :E]
    pred_actual = x[pred_idx + E - 1 + tp]

    predictions = np.zeros(len(pred_idx))

    for i, pv in enumerate(pred_vecs):
        # Distances to all library points
        dists = np.sqrt(np.sum((lib_vecs - pv)**2, axis=1))
        # Find E+1 nearest neighbors
        nn_idx = np.argsort(dists)[:E+1]
        nn_dists = dists[nn_idx]

        # Weights: exponential decay
        min_dist = nn_dists[0]
        if min_dist == 0:
            min_dist = 1e-10
        weights = np.exp(-nn_dists / min_dist)
        weights /= weights.sum()

        predictions[i] = np.dot(weights, lib_targets[nn_idx])

    return pred_actual, predictions

# --- Sweep E for logistic map ---
x_log = logistic_map(r=3.9, n=2000)
E_range = range(1, 11)
rhos = []

for E in E_range:
    obs, pred = simplex_projection(x_log, E=E, tp=1)
    rho, _ = pearsonr(obs, pred)
    rhos.append(rho)

fig, axes = plt.subplots(1, 2, figsize=(14, 4))

axes[0].plot(list(E_range), rhos, marker="o", linewidth=2, color="#2c3e50")
axes[0].set_title("Simplex: Forecast Skill vs $E$ (Logistic Map)", fontsize=12)
axes[0].set_xlabel("Embedding dimension $E$")
axes[0].set_ylabel("Correlation $\\rho$")
axes[0].grid(True, alpha=0.3)

# Best E
E_best = list(E_range)[np.argmax(rhos)]
obs_best, pred_best = simplex_projection(x_log, E=E_best, tp=1)
axes[1].scatter(obs_best, pred_best, s=3, alpha=0.4, color="#e74c3c")
axes[1].plot([0, 1], [0, 1], "k--", alpha=0.3)
axes[1].set_title(f"Predicted vs Observed ($E={E_best}$, $\\rho={max(rhos):.3f}$)", fontsize=12)
axes[1].set_xlabel("Observed $x_{t+1}$")
axes[1].set_ylabel("Predicted $\\hat{{x}}_{{t+1}}$")
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# --- Forecast skill decay with horizon (chaos signature) ---
horizons = range(1, 11)
rho_by_h = []
for h in horizons:
    obs_h, pred_h = simplex_projection(x_log, E=E_best, tp=h)
    r, _ = pearsonr(obs_h, pred_h)
    rho_by_h.append(r)

plt.figure(figsize=(10, 4))
plt.plot(list(horizons), rho_by_h, marker="o", linewidth=2, color="#e74c3c")
plt.title(f"Forecast Skill Decay with Horizon (Logistic, $E={E_best}$)", fontsize=12)
plt.xlabel("Prediction horizon $h$")
plt.ylabel("Correlation $\\rho$")
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print(f"Optimal E = {E_best} (logistic map has 1D attractor)")
print(f"Rho decays rapidly with horizon -> signature of chaos")
print(f"Short-term predictability + long-term unpredictability = deterministic chaos")


## **PART 4 — S-maps (Nonlinearity Detection)**

> **Why this part matters**
>
> Simplex projection uses nearest neighbors with fixed weights.
> **S-maps** (Sequential Locally Weighted Global Linear Maps) generalize this
> by fitting **local linear models** with a tuning parameter $\theta$ that
> controls the degree of localization.

---

### **4.1 The Algorithm**

For each prediction point $\mathbf{v}_T$:

1. Compute distances $d_i = \|\mathbf{v}_T - \mathbf{v}_i\|$ to all library points
2. Assign weights $w_i = \exp(-\theta \cdot d_i / \bar{d})$ where $\bar{d}$ is the mean distance
3. Fit a **weighted linear regression**: $x_{t+1} = \mathbf{c}'\mathbf{v}_t + c_0$ using weights $w_i$
4. Predict: $\hat{x}_{T+1} = \mathbf{c}'\mathbf{v}_T + c_0$

---

### **4.2 The $\theta$ Parameter**

$\theta$ controls the **locality** of the linear fit:

| $\theta$ | Behavior | Equivalent to |
|---|---|---|
| $\theta = 0$ | All points weighted equally | Global linear model (like AR) |
| $\theta > 0$ | Nearby points weighted more | Local (state-dependent) linear model |
| Large $\theta$ | Only nearest neighbors matter | Strongly nonlinear |

---

### **4.3 Nonlinearity Detection**

If optimal forecast skill occurs at $\theta > 0$: evidence of **nonlinearity**.
If $\theta = 0$ is best: the system is effectively **linear** (AR-like).

> **Exam-safe statement:** "S-maps test for nonlinearity by comparing global ($\theta=0$)
> vs local ($\theta > 0$) linear models. If local models forecast better,
> the dynamics are state-dependent (nonlinear)."


In [ ]:
# ============================================================
# PART 4: S-maps — Nonlinearity Detection
# ============================================================

def smap(x, E, theta, tp=1, lib_range=None, pred_range=None):
    """
    S-map (Sugihara 1994). Locally weighted linear regression.
    theta=0: global linear (AR-like). theta>0: local (nonlinear).
    """
    M = embed(x, E + 1, tau=1)
    n_emb = M.shape[0]

    if lib_range is None:
        lib_range = (0, n_emb // 2)
    if pred_range is None:
        pred_range = (n_emb // 2, n_emb - tp)

    lib_idx = np.arange(lib_range[0], min(lib_range[1], n_emb - tp))
    pred_idx = np.arange(pred_range[0], min(pred_range[1], n_emb - tp))

    lib_vecs = M[lib_idx, :E]
    lib_targets = x[lib_idx + E - 1 + tp]

    pred_vecs = M[pred_idx, :E]
    pred_actual = x[pred_idx + E - 1 + tp]

    predictions = np.zeros(len(pred_idx))

    for i, pv in enumerate(pred_vecs):
        dists = np.sqrt(np.sum((lib_vecs - pv)**2, axis=1))
        d_mean = np.mean(dists)
        if d_mean == 0:
            d_mean = 1e-10

        weights = np.exp(-theta * dists / d_mean)

        # Weighted linear regression: target = c0 + c1*v1 + ... + cE*vE
        W = np.diag(np.sqrt(weights))
        X_reg = np.column_stack([np.ones(len(lib_idx)), lib_vecs])
        try:
            Xw = W @ X_reg
            yw = W @ lib_targets
            coeffs = np.linalg.lstsq(Xw, yw, rcond=None)[0]
            pv_aug = np.concatenate([[1.0], pv])
            predictions[i] = pv_aug @ coeffs
        except Exception:
            predictions[i] = np.nan

    mask = ~np.isnan(predictions)
    return pred_actual[mask], predictions[mask]

# --- Compare theta values on logistic map (nonlinear) ---
thetas = [0, 0.5, 1.0, 2.0, 4.0, 8.0]
E_log = 2
rhos_smap = []

for theta in thetas:
    obs_s, pred_s = smap(x_log, E=E_log, theta=theta, tp=1)
    rho_s, _ = pearsonr(obs_s, pred_s)
    rhos_smap.append(rho_s)

# --- Compare on AR(1) (linear) ---
np.random.seed(42)
x_ar = np.zeros(2000)
for t in range(1, 2000):
    x_ar[t] = 0.7 * x_ar[t-1] + np.random.normal(0, 1)

rhos_ar = []
for theta in thetas:
    obs_a, pred_a = smap(x_ar, E=2, theta=theta, tp=1)
    rho_a, _ = pearsonr(obs_a, pred_a)
    rhos_ar.append(rho_a)

fig, axes = plt.subplots(1, 2, figsize=(14, 4))

axes[0].plot(thetas, rhos_smap, marker="o", linewidth=2, color="#e74c3c", label="Logistic (chaotic)")
axes[0].plot(thetas, rhos_ar, marker="s", linewidth=2, color="#2980b9", label="AR(1) (linear)")
axes[0].set_title("S-map: Forecast Skill vs $\\theta$", fontsize=12)
axes[0].set_xlabel("$\\theta$ (localization)")
axes[0].set_ylabel("Correlation $\\rho$")
axes[0].legend(fontsize=10)
axes[0].grid(True, alpha=0.3)

axes[1].bar(range(len(thetas)), np.array(rhos_smap) - np.array(rhos_smap)[0],
            tick_label=[str(t) for t in thetas], color="#e74c3c", alpha=0.7, label="Logistic")
axes[1].bar(range(len(thetas)), np.array(rhos_ar) - np.array(rhos_ar)[0],
            tick_label=[str(t) for t in thetas], color="#2980b9", alpha=0.4, label="AR(1)")
axes[1].set_title("Improvement over $\\theta=0$ (global linear)", fontsize=12)
axes[1].set_xlabel("$\\theta$")
axes[1].set_ylabel("$\\Delta\\rho$")
axes[1].axhline(0, color="gray", linestyle="--")
axes[1].legend(fontsize=10)
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("Nonlinearity test:")
print(f"  Logistic: best theta = {thetas[np.argmax(rhos_smap)]} (theta>0 wins -> NONLINEAR)")
print(f"  AR(1):    best theta = {thetas[np.argmax(rhos_ar)]} (theta~0 wins -> LINEAR)")


## **PART 5 — Convergent Cross-Mapping (CCM)**

> **Why this part matters**
>
> Granger causality (Week 5) tests whether past values of $X$ improve **linear predictions**
> of $Y$ in a stochastic framework. CCM (Sugihara et al., 2012) detects causality
> in **deterministic dynamical systems** using attractor reconstruction.
>
> This is especially relevant for your research on coupled environmental systems.

---

### **5.1 The Core Idea**

If $X$ **causally influences** $Y$ in a dynamical system, then:
- $X$'s information is embedded in $Y$'s dynamics (because $Y$ is driven by $X$)
- $Y$'s reconstructed attractor $M_Y$ contains information about $X$
- We can **cross-map**: use $M_Y$ to estimate $X$ (not the other way!)

> **Counterintuitive direction:** To test "$X$ causes $Y$", we cross-map from $M_Y$ to $X$.
> If $X$ drives $Y$, then $Y$'s shadow manifold "knows about" $X$.

---

### **5.2 The Algorithm**

1. Embed $Y$ to reconstruct $M_Y$: $\mathbf{v}_t^{(Y)} = (Y_t, Y_{t-1}, \ldots, Y_{t-E+1})$
2. For each point $\mathbf{v}_t^{(Y)}$ in $M_Y$, find its $E+1$ nearest neighbors
3. Use the **same time indices** of these neighbors to estimate $X_t$ from $X$'s values
4. Compute correlation $\rho(X, \hat{X}|M_Y)$ — the **cross-map skill**

---

### **5.3 Convergence: The Key Test**

The signature of true causality is **convergence**:

$$
\rho(X, \hat{X}|M_Y) \text{ increases with library size } L
$$

As $L$ grows, the shadow manifold $M_Y$ becomes denser, and cross-map estimates improve.

- **Convergence** → $X$ causally influences $Y$
- **No convergence** → no dynamical causal link

---

### **5.4 Asymmetry**

CCM is inherently **asymmetric**: $X \to Y$ and $Y \to X$ are tested separately, and the strengths may differ. In coupled systems, both directions may show convergence but at different rates.

> **Exam-critical:** "CCM tests dynamical causality by cross-mapping: if $X$ causes $Y$,
> then $Y$'s shadow manifold contains information about $X$, and cross-map skill
> converges as library size increases."


In [ ]:
# ============================================================
# PART 5: CCM — Convergent Cross-Mapping
# ============================================================

def ccm(x, y, E, tp=0, lib_sizes=None, n_reps=20):
    """
    Convergent Cross-Mapping: test if X causes Y.
    Cross-maps from M_Y -> X (if X causes Y, M_Y contains info about X).
    Returns library sizes and mean cross-map correlation.
    """
    n = min(len(x), len(y))
    x, y = x[:n], y[:n]

    if lib_sizes is None:
        lib_sizes = np.unique(np.linspace(E+2, n - (E-1) - 1, 15).astype(int))

    results = {"lib_size": [], "rho_mean": [], "rho_std": []}

    for L in lib_sizes:
        rhos = []
        for rep in range(n_reps):
            # Random subsample of indices
            valid_idx = np.arange(E-1, n-1)
            if L > len(valid_idx):
                continue
            lib_idx = np.sort(np.random.choice(valid_idx, size=L, replace=False))

            # Embed Y
            M_y = embed(y, E, tau=1)
            max_idx = M_y.shape[0]
            lib_emb_idx = lib_idx[lib_idx < max_idx]
            if len(lib_emb_idx) < E + 2:
                continue

            lib_vecs = M_y[lib_emb_idx]

            # Cross-map: for each library point, find E+1 neighbors, predict X
            x_obs_list = []
            x_pred_list = []

            for i, idx in enumerate(lib_emb_idx):
                dists = np.sqrt(np.sum((lib_vecs - lib_vecs[i])**2, axis=1))
                dists[i] = np.inf  # exclude self
                nn = np.argsort(dists)[:E+1]
                nn_dists = dists[nn]
                min_d = nn_dists[0] if nn_dists[0] > 0 else 1e-10
                w = np.exp(-nn_dists / min_d)
                w /= w.sum()

                # Corresponding X values at the neighbor time indices
                nn_time_idx = lib_emb_idx[nn]
                x_pred_val = np.dot(w, x[nn_time_idx + E - 1])
                x_obs_val = x[idx + E - 1]

                x_obs_list.append(x_obs_val)
                x_pred_list.append(x_pred_val)

            if len(x_obs_list) > 3:
                r, _ = pearsonr(x_obs_list, x_pred_list)
                rhos.append(r)

        if rhos:
            results["lib_size"].append(L)
            results["rho_mean"].append(np.mean(rhos))
            results["rho_std"].append(np.std(rhos))

    return results

# --- Simulate coupled logistic maps ---
np.random.seed(123)
n_ccm = 3000

# Unidirectional: X drives Y, but Y does NOT drive X
rx, ry = 3.8, 3.5
beta_xy = 0.1  # X -> Y coupling

x_coupled = np.zeros(n_ccm)
y_coupled = np.zeros(n_ccm)
x_coupled[0], y_coupled[0] = 0.4, 0.2

for t in range(n_ccm - 1):
    x_coupled[t+1] = x_coupled[t] * (rx - rx * x_coupled[t])
    y_coupled[t+1] = y_coupled[t] * (ry - ry * y_coupled[t] - beta_xy * x_coupled[t])

# --- Run CCM in both directions ---
E_ccm = 2
print("Running CCM (this may take a moment)...")
ccm_x_causes_y = ccm(x_coupled, y_coupled, E=E_ccm, n_reps=30)  # cross-map M_Y -> X
ccm_y_causes_x = ccm(y_coupled, x_coupled, E=E_ccm, n_reps=30)  # cross-map M_X -> Y

fig, ax = plt.subplots(1, 1, figsize=(10, 5))

ax.errorbar(ccm_x_causes_y["lib_size"], ccm_x_causes_y["rho_mean"],
            yerr=ccm_x_causes_y["rho_std"], linewidth=2, marker="o",
            color="#e74c3c", label="$X$ causes $Y$? (cross-map $M_Y \\to X$)", capsize=3)

ax.errorbar(ccm_y_causes_x["lib_size"], ccm_y_causes_x["rho_mean"],
            yerr=ccm_y_causes_x["rho_std"], linewidth=2, marker="s",
            color="#2980b9", label="$Y$ causes $X$? (cross-map $M_X \\to Y$)", capsize=3)

ax.set_title("CCM: Convergence Test (Unidirectional $X \\to Y$)", fontsize=13)
ax.set_xlabel("Library size $L$")
ax.set_ylabel("Cross-map correlation $\\rho$")
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print(f"\nExpected: X causes Y -> red line CONVERGES (increases with L)")
print(f"          Y does not cause X -> blue line does NOT converge or is flat")
print(f"\nX->Y final rho: {ccm_x_causes_y['rho_mean'][-1]:.3f}")
print(f"Y->X final rho: {ccm_y_causes_x['rho_mean'][-1]:.3f}")


## **PART 6 — Granger vs CCM: When to Use Which**

> **Why this part matters**
>
> Both Granger causality and CCM test for causal relationships in time series,
> but they operate under **fundamentally different assumptions**. Knowing when
> to use each is exam-critical.

---

### **6.1 Comparison Table**

| | Granger Causality | CCM |
|---|---|---|
| **Framework** | Stochastic (ARMA/VAR) | Deterministic dynamical systems |
| **Mechanism** | Predictive improvement | Attractor information sharing |
| **Test** | F-test on cross-lag coefficients | Convergence of cross-map skill with $L$ |
| **Assumes** | Stationarity, linearity, separability | Deterministic dynamics, weak-to-moderate coupling |
| **Fails when** | Nonlinear dynamics, synchronized systems | Pure stochastic processes, very strong coupling |
| **Direction** | Past of $X_2$ in equation for $X_1$ | $M_Y \to X$ tests "$X$ causes $Y$" (counterintuitive!) |
| **Null** | $H_0$: cross-lag coefficients = 0 | No convergence as $L$ increases |

---

### **6.2 When Granger Fails but CCM Works**

- **Nonlinear coupling**: $X$ influences $Y$ through nonlinear dynamics that linear VAR cannot capture
- **Mirage correlations**: in coupled dynamical systems, Granger can detect causality in the **wrong direction** (Sugihara et al., 2012)
- **Synchronized systems**: tightly coupled systems appear simultaneously predictable, confusing Granger tests

---

### **6.3 When CCM Fails but Granger Works**

- **Pure stochastic systems**: CCM requires deterministic structure; purely stochastic processes have no attractor to reconstruct
- **Very strong coupling**: when systems are perfectly synchronized, cross-map skill saturates immediately (no convergence)
- **Short time series**: CCM needs enough data for attractor reconstruction; Granger needs fewer observations

---

### **6.4 Best Practice**

> Use **both methods**. If both agree, confidence increases.
> If they disagree, the disagreement itself is informative about the nature of the system
> (stochastic vs deterministic, linear vs nonlinear).

> **Exam-safe statement:** "Granger causality tests linear predictive improvement in stochastic systems.
> CCM tests dynamical information sharing via attractor reconstruction.
> They are complementary, not competing."


## **PART 7 — Modern ML Methods: A Brief Overview**

> **Why this part matters**
>
> Modern machine learning approaches — particularly recurrent neural networks (RNNs)
> and Long Short-Term Memory (LSTM) networks — have gained prominence in time series
> forecasting. Understanding their conceptual role relative to classical methods is sufficient for the exam.

---

### **7.1 RNN (Recurrent Neural Network)**

RNNs process sequences by maintaining a **hidden state** $\mathbf{h}_t$ that is updated at each time step:

$$
\mathbf{h}_t = f(\mathbf{W}_h \mathbf{h}_{t-1} + \mathbf{W}_x \mathbf{x}_t + \mathbf{b})
$$

**Connection to state space:** The hidden state $\mathbf{h}_t$ is analogous to the Kalman filter's state vector — both summarize past information for prediction.

**Problem:** Vanilla RNNs suffer from **vanishing/exploding gradients** during training — they struggle with long-range dependencies.

---

### **7.2 LSTM (Long Short-Term Memory)**

LSTMs solve the gradient problem via **gating mechanisms**:

- **Forget gate**: decides what to remove from memory
- **Input gate**: decides what new information to store
- **Output gate**: decides what to expose from memory

This allows LSTMs to learn dependencies across long sequences.

---

### **7.3 Classical vs ML: When to Use Which**

| Criterion | Classical (ARIMA, VAR, SS) | ML (RNN/LSTM) |
|---|---|---|
| **Interpretability** | High (parameters have meaning) | Low (black box) |
| **Data requirement** | Small-to-moderate ($n < 1000$) | Large ($n > 10{,}000$) |
| **Uncertainty quantification** | Built-in (PIs, Kalman covariance) | Requires extra work |
| **Nonlinearity** | Limited (unless S-maps/CCM) | Naturally nonlinear |
| **Stationarity** | Assumed/enforced | Not required |
| **Exam relevance** | CRITICAL | Conceptual understanding sufficient |

> **Exam-safe statement:** "RNNs/LSTMs are nonlinear function approximators for sequences.
> They can capture complex patterns but require large data, lack interpretability,
> and do not natively produce calibrated prediction intervals.
> Classical methods remain preferred when data is limited and interpretability matters."


## **PART 8 — Synthesis (Exam-Ready)**

> **Core points you must be able to state without hesitation:**

---

### **Nonlinear Dynamics**

- Chaotic systems are deterministic but practically unpredictable at long horizons
- ACF of chaotic data can look like white noise — linear tools miss the structure entirely
- Lag plots and attractor reconstruction reveal hidden deterministic patterns

### **Takens' Theorem**

- A single observed variable can reconstruct the full attractor via time-delay embedding
- Embedding dimension $E$ and delay $\tau$ are the two parameters
- The shadow attractor preserves the topology and dynamics of the original system

### **Simplex Projection**

- Nearest-neighbor forecasting on the reconstructed attractor
- Optimal $E$ reflects attractor dimensionality
- Forecast skill decay rate distinguishes chaos (rapid) from noise (gradual)

### **S-maps**

- Locally weighted linear regression on the attractor
- $\theta = 0$: global linear (AR-like); $\theta > 0$: local (nonlinear)
- If $\theta > 0$ gives better forecasts → evidence of nonlinearity

### **CCM**

- Tests dynamical causality via cross-mapping between shadow manifolds
- **Direction is counterintuitive**: to test "$X$ causes $Y$", cross-map from $M_Y$ to $X$
- The signature of true causality is **convergence** of cross-map skill with library size $L$
- Complementary to Granger causality — use both

### **Classical vs ML**

- Classical methods: interpretable, small-data, uncertainty quantification, theory-driven
- ML methods: flexible, large-data, nonlinear, but black-box
- For PhD exam: classical understanding is CRITICAL; ML is conceptual awareness


## **PART 9 — Self-Test: Exam Preparation Questions**

> Work through these without looking at notes first.

---

### **Conceptual Questions (Oral Exam Style)**

**Q1.** What is deterministic chaos? How can a deterministic system produce a time series that looks random?

**Q2.** State Takens' embedding theorem informally. What does it guarantee about the shadow attractor? What parameters must be chosen?

**Q3.** Explain simplex projection. How does it use nearest neighbors on the reconstructed attractor for forecasting?

**Q4.** How do you select the optimal embedding dimension $E$ in practice? What does the optimal $E$ tell you about the system?

**Q5.** Explain the S-map nonlinearity test. What does the $\theta$ parameter control? How do you interpret $\theta = 0$ being optimal vs $\theta > 0$?

**Q6.** Explain CCM step by step. Why is the cross-map direction counterintuitive (to test "$X$ causes $Y$", you map from $M_Y$ to $X$)?

**Q7.** What is the convergence criterion in CCM? Why is convergence — not just high cross-map skill — the signature of causality?

**Q8.** Compare Granger causality and CCM across five dimensions: framework, mechanism, assumptions, failure modes, and test procedure.

**Q9.** In what scenario would Granger causality detect causality in the wrong direction? How does CCM handle this?

**Q10.** Your colleague uses an LSTM to forecast air quality and says "we don't need ARIMA anymore." Give three reasons why this claim is problematic.

---

### **Technical Questions**

**Q11.** For the logistic map $x_{t+1} = 3.9\, x_t(1 - x_t)$:
- What is the true attractor dimension?
- What embedding dimension $E$ would simplex projection find optimal?
- Why does the ACF look like white noise despite the process being deterministic?

**Q12.** In a coupled system where $X$ drives $Y$ unidirectionally:
- Draw what the CCM convergence plot should look like (both directions)
- What happens if coupling strength increases toward synchronization?

---

### **Practical Challenge**

**Q13.** Modify the CCM simulation to make coupling **bidirectional** ($X \to Y$ and $Y \to X$ with different strengths). How do the convergence curves change?

**Q14.** Apply simplex projection to the Lorenz $x(t)$ series. What optimal $E$ do you find? Does forecast skill decay match the chaos signature?
